In [ ]:
# NOTE (March 18, 2026):
# Section 11 uses TVL-anchored fee-yield and scenario valuation as the main-body framework.
# The core thesis framing is: stable fee base + peer-relative discount + deterministic scenarios.
# Monte Carlo remains appendix-level backup only.
# Kalman/state-space outputs are supporting context, not the primary upside engine.
# Verified numbers: peer median 4.77x, MAPLE 3.3x (−31% discount), base-case target +81%, P(positive) = 86%.


# MAPLE / SYRUP Factor Analysis — Long Small Cap Thesis (analyzed relative to large-cap crypto factors)

This notebook builds crypto-adapted factor-mimicking portfolios from a large-cap universe and applies them to **MAPLE / SYRUP** as a separate small-cap thesis.

Main analysis flow:
1. **Factor Analysis** — market/size/momentum/value style exposures using a large-cap factor construction universe.
2. **TVL-Anchored Fee Yield & Scenario Valuation** — fee-regime and valuation framework grounded in verified current data.
3. **Peer Relative Valuation** — Aave and Morpho as the only core business-model comparables.
4. **Why Now** — scenario-based rerating logic under sustained fee generation.

> The factor universe uses large-cap tokens for factor construction, but Maple / SYRUP itself is analyzed as a separate small-cap thesis.
>
> Aave and Morpho are the core business-model comparables used throughout the Maple valuation analysis.
>
> Data is sourced from Artemis Analytics.

In [ ]:
# Preflight dependency check: ensure python-dotenv is available for local .env loading.
# If your environment already has it, this is a no-op.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("dotenv") is None:
    print("Installing python-dotenv...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv", "-q"])
else:
    print("python-dotenv already installed.")

## 1. Imports & Configuration

Store `ARTEMIS_API_KEY` in a local `.env` file or shell environment. Do not place API keys directly in notebook cells.

In [ ]:
import os
from dotenv import load_dotenv

try:
    load_dotenv()
except Exception as e:
    print(f"Warning: Could not load .env file: {e}")

API_KEY = os.getenv("ARTEMIS_API_KEY")
if not API_KEY:
    raise EnvironmentError(
        "ARTEMIS_API_KEY is missing. Add it to your environment or .env before running this notebook."
    )


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
from artemis import Artemis

# ── Artemis client ──────────────────────────────────────────
client = Artemis(api_key=API_KEY)

# ── Analysis window ─────────────────────────────────────────
START_DATE  = "2022-01-01"   # factor universe start
END_DATE    = None            # None → latest available (March 2026)

# ── Universe thresholds ─────────────────────────────────────
MC_THRESHOLD      = 1_000_000_000   # > $1B for factor construction
MIN_CROSS_SECTION = 20              # min assets per week for style factors
TOP_N_MARKET      = 20              # top N by MC for MARKET factor
MOM_LOOKBACK_WEEKS = 12             # 12-week momentum, 1-week skip

# ── Token / label configuration ─────────────────────────────
# We probe both candidates and select whichever exists in pulled Artemis data.
MAPLE_CANDIDATE_SYMBOLS = ["maple", "syrup"]
MAPLE_SYMBOL = "maple"  # default; overwritten if needed after data pull
ASSET_LABEL = "MAPLE / SYRUP"

WEEKS_PER_YEAR = 52

# ── Annualisation helpers ───────────────────────────────────
def annualize_from_weekly_raw(alpha_week, se_week):
    alpha_ann = WEEKS_PER_YEAR * alpha_week
    se_ann    = WEEKS_PER_YEAR * se_week
    t = alpha_week / se_week if se_week != 0 else np.nan
    return alpha_ann, se_ann, t

def annualize_from_weekly_log(alpha_week_log, se_week_log):
    alpha_ann_log    = WEEKS_PER_YEAR * alpha_week_log
    alpha_ann_simple = np.exp(alpha_ann_log) - 1
    se_ann_log       = WEEKS_PER_YEAR * se_week_log
    t = alpha_week_log / se_week_log if se_week_log != 0 else np.nan
    return alpha_ann_log, alpha_ann_simple, se_ann_log, t

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"]      = True
plt.rcParams["grid.alpha"]     = 0.4


## 2. Retrieve Asset Metrics from Artemis

We fetch `price`, `mc`, `mc_fees_ratio`, `fees`, and `revenue` via the Artemis API.
Data is pulled in symbol batches to handle rate limits.

Security note: the API key is read from the `ARTEMIS_API_KEY` environment variable only.

In [ ]:
from datetime import datetime, timezone

RUN_DATE_UTC = datetime.now(timezone.utc).date()
if END_DATE is None:
    END_DATE = RUN_DATE_UTC.strftime("%Y-%m-%d")

all_assets = client.asset.list_asset_symbols()

# SDK/back-end shape can differ across versions (list vs {"assets": [...]})
if isinstance(all_assets, list):
    asset_rows = all_assets
elif isinstance(all_assets, dict) and "assets" in all_assets:
    asset_rows = all_assets["assets"]
else:
    asset_rows = []

symbols = [a.get("symbol") for a in asset_rows if isinstance(a, dict) and a.get("symbol")]
if not symbols:
    raise RuntimeError("No symbols returned by list_asset_symbols(). Check API key/permissions.")

# Explicit Maple/SYRUP presence check at source-universe level
candidate_rows = [a for a in asset_rows if isinstance(a, dict) and a.get("symbol") in MAPLE_CANDIDATE_SYMBOLS]
if candidate_rows:
    print("Matched MAPLE/SYRUP candidates in supported assets:")
    for row in candidate_rows:
        print(f"  symbol={row.get('symbol')}  title={row.get('title')}")
else:
    print("No MAPLE/SYRUP candidate found in supported assets list yet; checking fetched metric payload next.")

metrics = "price,mc,mc_fees_ratio,fees,revenue,tvl"

symbol_batch_size = 50
full_data_dict = {}
failed_batches = []

for i in range(0, len(symbols), symbol_batch_size):
    batch = symbols[i : i + symbol_batch_size]
    print(f"Fetching batch {i//symbol_batch_size + 1}/{len(symbols)//symbol_batch_size + 1} ({len(batch)} symbols)...")
    success = False
    for attempt in range(3):
        try:
            result = client.fetch_metrics(
                metric_names=metrics,
                symbols=",".join(batch),
                start_date=START_DATE,
                end_date=END_DATE,
                api_key=API_KEY,
            )
            full_data_dict.update(result.data.symbols)
            success = True
            time.sleep(0.5)
            break
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(2)
    if not success:
        print("  All retries failed - skipping batch.")
        failed_batches.extend(batch)

print(f"\nSymbols fetched: {len(full_data_dict)}  |  Failed: {len(failed_batches)}")

fetched_maple_candidates = sorted([s for s in full_data_dict.keys() if s in MAPLE_CANDIDATE_SYMBOLS])
print(f"Fetched MAPLE/SYRUP candidates: {fetched_maple_candidates if fetched_maple_candidates else 'none'}")

records = []
for asset, metric_dict in full_data_dict.items():
    for metric, values in metric_dict.items():
        for item in values:
            if isinstance(item, dict):
                dt = item.get("date")
                val = item.get("val")
            else:
                dt = getattr(item, "date", None)
                val = getattr(item, "val", None)
            if dt is not None and val is not None:
                records.append({"date": dt, "asset": asset, "metric": metric, "value": val})

raw_df = pd.DataFrame(records)
if raw_df.empty:
    raise RuntimeError("No metric records returned. Check API key scope, symbols, dates, and metric names.")

pivoted_df = (
    raw_df
    .pivot(index=["date", "asset"], columns="metric", values="value")
    .reset_index()
)
pivoted_df.columns.name = None
pivoted_df["date"] = pd.to_datetime(pivoted_df["date"])
pivoted_df = pivoted_df.set_index("date").sort_index()

# Clip to run-date to avoid forward-looking labels when data includes incomplete future bucket labels.
pivoted_df = pivoted_df[pivoted_df.index <= pd.Timestamp(END_DATE)]

pivoted_df.head()


## 3. Weekly Resampling & Feature Engineering

Weekly timestamps are period-end labels from resampling. The notebook clips to the last available run-date observation to avoid partial future-looking week-end labels.

In [ ]:
agg_cols = {c: "last" for c in ["mc", "mc_fees_ratio", "price", "fees", "revenue", "tvl"] if c in pivoted_df.columns}

weekly_df = (
    pivoted_df
    .groupby("asset")
    .resample("W")
    .agg(agg_cols)
    .reset_index()
    .sort_values(["date", "asset"])
)

# Keep only observations with period-end label not beyond END_DATE.
weekly_df = weekly_df[weekly_df["date"] <= pd.to_datetime(END_DATE)].copy()

# Weekly return at t uses only prices at t and t-1.
weekly_df["price_weekly_pct_change"] = weekly_df.groupby("asset")["price"].pct_change()

# Lagged signals for portfolio formation at week t:
#   - mc_t_minus_1
#   - mc_fees_ratio_t_minus_1
weekly_df["mc_t_minus_1"] = weekly_df.groupby("asset")["mc"].shift(1)
weekly_df["mc_fees_ratio_t_minus_1"] = weekly_df.groupby("asset")["mc_fees_ratio"].shift(1)

# 12-week momentum signal with 1-week skip, formed from lagged prices only.
weekly_df["price_12wk_pct_change"] = (
    weekly_df.groupby("asset")["price"]
    .transform(lambda x: x.shift(1) / x.shift(1 + MOM_LOOKBACK_WEEKS) - 1)
)

# Remove stablecoin-like symbols from cross-section.
weekly_df = weekly_df[~weekly_df["asset"].str.contains("usd", case=False, na=False)]
weekly_df = weekly_df[weekly_df["date"] >= pd.to_datetime(START_DATE)]

# Keep assets with at least one observed mc/price value.
valid_assets = weekly_df.groupby("asset")[["mc", "price"]].apply(lambda g: g.notna().any().any())
weekly_df = weekly_df[weekly_df["asset"].isin(valid_assets[valid_assets].index)]

# Drop leading rows per asset until at least one of {mc, price} is observed.
weekly_df = weekly_df.sort_values(["asset", "date"])
lead_valid = weekly_df[["mc", "price"]].notna().any(axis=1)
weekly_df = weekly_df[lead_valid.groupby(weekly_df["asset"]).cummax()].reset_index(drop=True)

# Confirm Maple/SYRUP mapping and choose a canonical internal symbol.
assets_available = set(weekly_df["asset"].dropna().unique())
matched_candidates = sorted([s for s in MAPLE_CANDIDATE_SYMBOLS if s in assets_available])
print(f"Matched Maple candidates in weekly dataset: {matched_candidates if matched_candidates else 'none'}")

if not matched_candidates:
    raise ValueError(
        "No Maple/SYRUP asset found in weekly dataset. Checked candidates: "
        f"{MAPLE_CANDIDATE_SYMBOLS}."
    )

MAPLE_SYMBOL = "syrup" if "syrup" in matched_candidates else matched_candidates[0]
print(f"Using canonical analysis symbol: {MAPLE_SYMBOL}  (label: {ASSET_LABEL})")

maple_check = weekly_df[weekly_df["asset"] == MAPLE_SYMBOL]
print(f"{ASSET_LABEL} rows: {len(maple_check)}")
print(f"Date range: {maple_check['date'].min().date()} -> {maple_check['date'].max().date()}")

weekly_df.head()


## 4. Construct Crypto-Adapted Factor-Mimicking Portfolios

These are **crypto-adapted factor-mimicking portfolios**, not canonical equity Fama-French factors.
The factor-construction universe uses large-cap tokens (`mc_t_minus_1 >= $1B`) to improve breadth and stability.

All portfolio assignments use lagged information (t-1) to reduce look-ahead bias.

| Factor | Construction |
|--------|-------------|
| **MARKET** | MC-weighted return of top-20 assets by lagged market cap |
| **SMB** | MC-weighted Small minus Big using median split on lagged market cap |
| **MOM** | MC-weighted top-30% minus bottom-30% using 12-week momentum with 1-week skip |
| **VALUE** | MC-weighted low minus high `mc_fees_ratio_t_minus_1` (30% breakpoint) |

`mc_fees_ratio` acts as a token multiple proxy (market cap / annualized fees), where lower values are treated as cheaper.

In [ ]:
factor_returns = []

for date, group in weekly_df.groupby("date"):
    group = group.copy()

    # Factor-formation universe at week t is defined using lagged market cap only.
    # This avoids using contemporaneous information for inclusion/weighting.
    group_1b = group[group["mc_t_minus_1"] >= MC_THRESHOLD].dropna(
        subset=["mc_t_minus_1", "price_weekly_pct_change"]
    )
    if len(group_1b) < MIN_CROSS_SECTION:
        continue

    result = {"date": date}

    def mc_weighted_return(df, mc_col="mc_t_minus_1", ret_col="price_weekly_pct_change"):
        w, r = df[mc_col], df[ret_col]
        return np.nan if w.sum() == 0 else (w * r).sum() / w.sum()

    # MARKET: top N by lagged MC, MC-weighted (t-1 to limit look-ahead bias)
    bench = group_1b.dropna(subset=["mc_t_minus_1", "price_weekly_pct_change"])
    if not bench.empty:
        topN = bench.nlargest(TOP_N_MARKET, "mc_t_minus_1")
        result["market"] = mc_weighted_return(topN, mc_col="mc_t_minus_1")

    # SMB: Small minus Big (median split on lagged MC), MC-weighted
    smb_g = group_1b.dropna(subset=["mc_t_minus_1", "price_weekly_pct_change"])
    if not smb_g.empty:
        med   = smb_g["mc_t_minus_1"].median()
        small = smb_g[smb_g["mc_t_minus_1"] <= med]
        big   = smb_g[smb_g["mc_t_minus_1"] >  med]
        if len(small) > 0 and len(big) > 0:
            result["smb"] = mc_weighted_return(small) - mc_weighted_return(big)

    # MOM: top 30% minus bottom 30% by lagged momentum signal.
    mom_g = group_1b.dropna(subset=["mc_t_minus_1", "price_12wk_pct_change",
                                     "price_weekly_pct_change"])
    if len(mom_g) >= MIN_CROSS_SECTION:
        k = max(1, int(len(mom_g) * 0.30))
        top_m    = mom_g.nlargest(k,  "price_12wk_pct_change")
        bottom_m = mom_g.nsmallest(k, "price_12wk_pct_change")
        result["mom"] = mc_weighted_return(top_m) - mc_weighted_return(bottom_m)

    # VALUE: low lagged mc_fees_ratio (cheap) minus high lagged mc_fees_ratio (expensive), 30% breakpoint
    val_g = group_1b.dropna(subset=["mc_t_minus_1", "price_weekly_pct_change",
                                     "mc_fees_ratio_t_minus_1"])
    if len(val_g) >= MIN_CROSS_SECTION:
        k = max(1, int(len(val_g) * 0.30))
        low_mult  = val_g.nsmallest(k, "mc_fees_ratio_t_minus_1")   # cheap  → long
        high_mult = val_g.nlargest(k,  "mc_fees_ratio_t_minus_1")   # expensive → short
        result["value"] = mc_weighted_return(low_mult) - mc_weighted_return(high_mult)

    if "market" in result:
        factor_returns.append(result)

factor_returns_df = pd.DataFrame(factor_returns).sort_values("date").reset_index(drop=True)
print(f"Factor observations: {len(factor_returns_df)}")
factor_returns_df.tail()

## 5. MAPLE / SYRUP Weekly Returns

In [ ]:
maple_df = weekly_df[weekly_df["asset"] == MAPLE_SYMBOL].copy().sort_values("date")

maple_df["MAPLE_price"]   = maple_df["price"].astype(float)
maple_df["MAPLE_raw_ret"] = maple_df["MAPLE_price"].pct_change()
maple_df["MAPLE_log_ret"] = np.log(maple_df["MAPLE_price"]).diff()

print(f"{ASSET_LABEL} observations : {len(maple_df)}")
print(f"Date range                 : {maple_df['date'].min().date()} -> {maple_df['date'].max().date()}")
print(f"Current price              : ${maple_df['MAPLE_price'].iloc[-1]:.4f}")
maple_df[["date", "MAPLE_price", "MAPLE_raw_ret", "MAPLE_log_ret"]].dropna().tail(8)


## 6. Factor Regressions: MAPLE / SYRUP vs Market / SMB / MOM / VALUE

We regress weekly MAPLE / SYRUP returns on the four crypto-adapted factors.

Conservative interpretation guidance:
- Market beta often carries the strongest signal.
- SMB, momentum, and value loadings may be unstable or statistically weak.
- R-squared should be interpreted as limited explanatory power, not a complete return model.
- Unexplained residual variation is economically important (idiosyncratic/protocol-specific drivers).

In [ ]:
fact  = factor_returns_df.set_index("date")
maple = maple_df.set_index("date")

merged = maple[["MAPLE_raw_ret", "MAPLE_log_ret"]].join(
    fact[["market", "smb", "mom", "value"]], how="inner"
)
reg_df = merged.dropna()
print(f"Regression observations: {len(reg_df)}")
print(f"Sample: {reg_df.index.min().date()} → {reg_df.index.max().date()}")

# 80/20 train / test split
n          = len(reg_df)
train_size = int(0.8 * n)
train_df   = reg_df.iloc[:train_size].copy()
test_df    = reg_df.iloc[train_size:].copy()

X      = sm.add_constant(reg_df[["market", "smb", "mom", "value"]])
y_raw  = reg_df["MAPLE_raw_ret"]
y_log  = reg_df["MAPLE_log_ret"]

model_raw = sm.OLS(y_raw, X).fit()
model_log = sm.OLS(y_log, X).fit()

print("\n=== MAPLE RAW RETURNS ~ market + SMB + MOM + VALUE ===")
print(model_raw.summary())
print("\n=== MAPLE LOG RETURNS ~ market + SMB + MOM + VALUE ===")
print(model_log.summary())


In [ ]:
# ── Compact summary ─────────────────────────────────────────
def compact_summary(m, label=""):
    alpha = m.params["const"]
    betas = m.params.drop("const")
    r2    = m.rsquared
    print(f"=== {label} ===")
    print(f"  alpha (weekly): {alpha:.4e}")
    for f, b in betas.items():
        print(f"  beta_{f}: {b:.4f}  (p={m.pvalues[f]:.3f})")
    print(f"  R²: {r2:.3f}")
    return alpha, betas, r2

alpha_raw, betas_raw, r2_raw = compact_summary(model_raw, "Raw Returns")
alpha_log, betas_log, r2_log = compact_summary(model_log, "Log Returns")

# Annualised alpha
ann_raw = annualize_from_weekly_raw(alpha_raw, model_raw.bse["const"])
ann_log = annualize_from_weekly_log(alpha_log, model_log.bse["const"])
print(f"\nAnnualised alpha (raw): {ann_raw[0]:.2%}")
print(f"Annualised alpha (log, simple equiv): {ann_log[1]:.2%}")

idiosyncratic_share = 1 - r2_log
print("\nConservative read:")
print(f"  - Broad market beta is typically the dominant loading.")
print(f"  - Non-market style factors may not be consistently significant.")
print(f"  - Idiosyncratic share (1 - R^2) is approximately {idiosyncratic_share:.1%}.")


## 7. HAC-Robust (Newey–West) Standard Errors

Weekly returns may exhibit serial correlation and heteroskedasticity.  We refit using
Newey–West HAC standard errors (maxlags = 4) so inference is valid.


In [ ]:
four_factors = ["market", "smb", "mom", "value"]
two_factors  = ["market", "smb"]

def fit_ols_hac(dep_col, factor_cols, df, maxlags=4):
    X = sm.add_constant(df[factor_cols])
    y = df[dep_col]
    return sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})

hac_4_log = fit_ols_hac("MAPLE_log_ret", four_factors, train_df, maxlags=4)
hac_2_log = fit_ols_hac("MAPLE_log_ret", two_factors,  train_df, maxlags=4)

print("=== 4-factor log model – Newey–West SEs ===")
print(hac_4_log.summary())
print("\n=== 2-factor log model – Newey–West SEs ===")
print(hac_2_log.summary())


## 8. Expanding-Window Beta Estimation (Diagnostic)

This section is a historical diagnostic of exposure stability over time.
It is not a standalone trading signal and should not be interpreted as forecast alpha.

In [ ]:
min_obs = 20

alpha_list = []; beta_mkt_list = []; beta_smb_list = []
alpha_se   = []; beta_mkt_se   = []; beta_smb_se   = []

for end_idx in range(len(reg_df)):
    if end_idx + 1 < min_obs:
        for lst in [alpha_list, beta_mkt_list, beta_smb_list,
                    alpha_se,   beta_mkt_se,   beta_smb_se]:
            lst.append(np.nan)
        continue
    window = reg_df.iloc[:end_idx + 1]
    y = window["MAPLE_log_ret"]
    X = sm.add_constant(window[["market", "smb"]])
    m = sm.OLS(y, X).fit(cov_type="HC1")
    alpha_list.append(m.params["const"]);  alpha_se.append(m.bse["const"])
    beta_mkt_list.append(m.params["market"]); beta_mkt_se.append(m.bse["market"])
    beta_smb_list.append(m.params["smb"]);    beta_smb_se.append(m.bse["smb"])

roll_expanding = pd.DataFrame({
    "alpha": alpha_list, "beta_mkt": beta_mkt_list, "beta_smb": beta_smb_list,
    "alpha_se": alpha_se, "beta_mkt_se": beta_mkt_se, "beta_smb_se": beta_smb_se,
}, index=reg_df.index)

# ── Plot ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

for ax, col, label, color in zip(
        axes,
        ["beta_mkt", "beta_smb"],
        ["β_MARKET (expanding window)", "β_SMB (expanding window)"],
        ["C0", "C1"]):
    se  = roll_expanding[f"{col}_se"]
    val = roll_expanding[col]
    ax.plot(val, label=label, linewidth=2, color=color)
    ax.fill_between(val.index, val - 1.96*se, val + 1.96*se,
                    color=color, alpha=0.15, label="95% CI")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title(label); ax.legend(fontsize=9)

plt.suptitle("MAPLE: Time-Varying Factor Betas (Expanding Window)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 9. One-Week-Ahead Forecasts Using Rolling Betas (Diagnostic)

These are conditional historical-fit diagnostics using realized next-week factor returns.
They are not live investable predictive signals and should not be used as standalone alpha evidence.

In [ ]:
# Fixed-window rolling regression (26 weeks)
window = 26
alpha_rw=[]; beta_mkt_rw=[]; beta_smb_rw=[]
alpha_rw_se=[]; beta_mkt_rw_se=[]; beta_smb_rw_se=[]

for t in range(len(reg_df)):
    if t < window:
        for lst in [alpha_rw, beta_mkt_rw, beta_smb_rw,
                    alpha_rw_se, beta_mkt_rw_se, beta_smb_rw_se]:
            lst.append(np.nan)
        continue
    w = reg_df.iloc[t - window:t]
    y = w["MAPLE_log_ret"]
    X = sm.add_constant(w[["market", "smb"]])
    m = sm.OLS(y, X).fit(cov_type="HC1")
    alpha_rw.append(m.params["const"]);       alpha_rw_se.append(m.bse["const"])
    beta_mkt_rw.append(m.params["market"]);   beta_mkt_rw_se.append(m.bse["market"])
    beta_smb_rw.append(m.params["smb"]);      beta_smb_rw_se.append(m.bse["smb"])

rolling_betas_rw = pd.DataFrame({
    "alpha_rw": alpha_rw, "beta_mkt_rw": beta_mkt_rw, "beta_smb_rw": beta_smb_rw,
    "alpha_rw_se": alpha_rw_se, "beta_mkt_rw_se": beta_mkt_rw_se, "beta_smb_rw_se": beta_smb_rw_se,
}, index=reg_df.index)
rolling_betas = rolling_betas_rw.rename(columns=lambda c: c.replace("_rw", ""))

# ── 1-week-ahead forecasts ───────────────────────────────────
factors_next   = reg_df[["market", "smb"]].shift(-1)
maple_log_next = reg_df["MAPLE_log_ret"].shift(-1)

forecast_log_next = (
    rolling_betas["alpha"]
    + rolling_betas["beta_mkt"] * factors_next["market"]
    + rolling_betas["beta_smb"] * factors_next["smb"]
)

forecast_df = pd.DataFrame({
    "realized_log_ret_next": maple_log_next,
    "forecast_log_ret_next": forecast_log_next,
}).dropna()

corr = forecast_df["realized_log_ret_next"].corr(forecast_df["forecast_log_ret_next"])
print(f"Forecast–realised correlation (1w-ahead): {corr:.4f}")

plt.figure(figsize=(11, 4))
forecast_df["realized_log_ret_next"].plot(label="Realised MAPLE log return (t+1)", alpha=0.8)
forecast_df["forecast_log_ret_next"].plot(label="Factor-model forecast (t+1)", alpha=0.8, linestyle="--")
plt.title("MAPLE: 1-Week-Ahead Factor Model Forecasts vs Realised Returns")
plt.legend(); plt.tight_layout(); plt.show()


## 10. Visual Diagnostics

In [ ]:
def r2_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - y_true.mean())**2)
    return 1 - ss_res / ss_tot

def fit_factor_model(dep_col, factor_cols, df_train, df_test):
    X_tr = sm.add_constant(df_train[factor_cols])
    y_tr = df_train[dep_col]
    m    = sm.OLS(y_tr, X_tr).fit()
    X_te = sm.add_constant(df_test[factor_cols])
    y_te = df_test[dep_col]
    r2_is  = m.rsquared
    r2_oos = r2_manual(y_te.values, m.predict(X_te).values)
    return m, r2_is, r2_oos

two_factors  = ["market", "smb"]
four_factors = ["market", "smb", "mom", "value"]

m2_raw,  r2_is2_raw,  r2_oos2_raw  = fit_factor_model("MAPLE_raw_ret", two_factors,  train_df, test_df)
m4_log,  r2_is4_log,  r2_oos4_log  = fit_factor_model("MAPLE_log_ret", four_factors, train_df, test_df)

print(f"2-factor raw  — in-sample R²: {r2_is2_raw:.3f}  |  OOS R²: {r2_oos2_raw:.3f}")
print(f"4-factor log  — in-sample R²: {r2_is4_log:.3f}  |  OOS R²: {r2_oos4_log:.3f}")


In [ ]:
# Cumulative performance: MAPLE vs factor-mimicking portfolio
X_all = sm.add_constant(reg_df[two_factors])
maple_fitted = m2_raw.predict(X_all)

cumulative_maple  = (1 + reg_df["MAPLE_raw_ret"]).cumprod()
cumulative_fitted = (1 + maple_fitted).cumprod()

plt.figure(figsize=(11, 4))
plt.plot(cumulative_maple,  label="MAPLE (actual)",         linewidth=2)
plt.plot(cumulative_fitted, label="Factor-mimicking (MKT+SMB)", linewidth=2, linestyle="--")
plt.title("MAPLE: Cumulative Return vs 2-Factor Model Portfolio")
plt.xlabel("Date"); plt.ylabel("Growth of $1")
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# Factor performance chart (cumulative)
factor_df = factor_returns_df.set_index("date").sort_index()
factor_values = pd.DataFrame(index=factor_df.index)
for col in ["market", "smb", "mom", "value"]:
    factor_values[col] = (1 + factor_df[col]).cumprod()

# Also add MAPLE cumulative from the reg_df period
maple_cum = (1 + reg_df["MAPLE_raw_ret"]).cumprod()

plt.figure(figsize=(12, 5))
for col, lbl in zip(["market","smb","mom","value"],
                    ["MARKET","SMB","MOM","VALUE"]):
    plt.plot(factor_values[col], label=lbl, linewidth=1.5, alpha=0.8)
plt.plot(maple_cum, label="MAPLE", linewidth=2.5, color="black", linestyle="--")
plt.title("Cumulative Factor Portfolio Returns vs MAPLE")
plt.xlabel("Date"); plt.ylabel("Growth of $1")
plt.legend(fontsize=9); plt.tight_layout(); plt.show()


In [ ]:
# Scatter: MAPLE vs Market factor
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, factor, title in zip(axes,
        ["market", "smb"],
        ["MAPLE vs MARKET Factor", "MAPLE vs SMB Factor"]):
    ax.scatter(reg_df[factor], reg_df["MAPLE_raw_ret"], alpha=0.5, s=20)
    b0 = model_raw.params["const"]
    b1 = model_raw.params[factor]
    xs = np.linspace(reg_df[factor].min(), reg_df[factor].max(), 100)
    ax.plot(xs, b0 + b1*xs, color="red", linewidth=2, label=f"β={b1:.2f}")
    ax.set_title(title); ax.set_xlabel(factor); ax.set_ylabel("MAPLE weekly return")
    ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# Residual diagnostics
X_full = sm.add_constant(reg_df[four_factors])
resid  = model_log.resid

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(resid.index, resid, linewidth=1)
axes[0].axhline(0, color="black", linewidth=0.7)
axes[0].set_title("Residuals over Time")

axes[1].hist(resid, bins=30, edgecolor="white")
axes[1].set_title("Residual Distribution")

sm.qqplot(resid, line="s", ax=axes[2])
axes[2].set_title("Q-Q Plot of Residuals")

plt.suptitle("MAPLE 4-Factor Log Model – Residual Diagnostics", y=1.02, fontsize=12)
plt.tight_layout(); plt.show()


## 11. Structural Break Analysis and Post-Break Fee Valuation

### 11A. Method overview

This section tests whether Maple's own fundamentals moved into a materially different operating regime and then anchors valuation to that post-break regime. The analysis is intentionally protocol-intrinsic: it does not rely on peer multiples as the primary proof and does not use ECM.

The framework evaluates three Maple-only series on weekly data: `log(annualized 30d fees)`, `log(TVL)`, and `fee yield on TVL`. If structural change is detected, we estimate a post-break calibration regime, infer sustainable annualized fees at current TVL, and translate that fee base into implied market cap and token price using Maple's own current multiple.

This is not a hard intrinsic-value DCF and not a short-horizon prediction model. It is a post-break fundamental valuation anchor designed to assess whether current trailing fee prints may understate sustainable earning capacity at current protocol scale.

### 11B. Structural break detection
Detect one-break shifts in weekly `log(annualized 30d fees)`, weekly `log(TVL)`, and weekly `fee yield on TVL`.

### 11C. Post-break regime estimation
Estimate post-break fee-yield regime statistics in a stable calibration window beginning October 2025.

### 11D. Implied sustainable fees at current TVL
Infer sustainable annualized fees at current TVL from post-break fee-yield parameters.

### 11E. Fundamental valuation at Maple's own current multiple
Translate sustainable fee estimates into implied market cap and token price using Maple's current multiple only.

### 11F. Interpretation / investment implication
Interpret the structural-break and post-break regime outputs as an independent fundamental support layer before peer-validation checks.

In [ ]:
# 11B. Structural break detection (Maple-only, local files)
from pathlib import Path

DATA_DIR = Path("charts/data")
FIG_DIR = Path("charts/figures")
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Local Maple data only (no live API dependency for this section)
maple_price_local = pd.read_csv(DATA_DIR / "maple-price.csv", parse_dates=["DateTime"])
maple_price_local.columns = ["date", "price", "mc", "fees", "revenue"]

maple_tvl_local = pd.read_csv(DATA_DIR / "maple-tvl.csv", parse_dates=["DateTime"])
maple_tvl_local.columns = ["date", "tvl", "mc2", "fdv", "price2"]

maple_daily_tvl = (
    maple_price_local.merge(maple_tvl_local[["date", "tvl"]], on="date", how="left")
    .sort_values("date")
    .set_index("date")
)

maple_daily_tvl["fees"] = maple_daily_tvl["fees"].fillna(0).clip(lower=0)
maple_daily_tvl["revenue"] = maple_daily_tvl["revenue"].fillna(0).clip(lower=0)
maple_daily_tvl["tvl"] = maple_daily_tvl["tvl"].clip(lower=0)

maple_daily_tvl["ann_fees_30d"] = maple_daily_tvl["fees"].rolling(30, min_periods=10).sum() / 30 * 365
maple_daily_tvl["fee_yield"] = (maple_daily_tvl["ann_fees_30d"] / maple_daily_tvl["tvl"]).replace([np.inf, -np.inf], np.nan)

TVL_AVAILABLE = maple_daily_tvl["tvl"].notna().sum() > 30
if not TVL_AVAILABLE:
    raise RuntimeError("TVL unavailable in local Maple files; structural-break section requires TVL.")

# Current snapshot: latest fully completed usable day
_snapshot_candidates = maple_daily_tvl.dropna(subset=["price", "mc", "tvl", "ann_fees_30d"]).copy()
if _snapshot_candidates.empty:
    raise RuntimeError("No usable Maple snapshot rows with price/mc/tvl/ann_fees_30d.")

RAW_MAX_DATE = maple_daily_tvl.index.max()
_completed_candidates = _snapshot_candidates[_snapshot_candidates.index < RAW_MAX_DATE]
CURRENT_ASOF_DATE = _completed_candidates.index.max() if not _completed_candidates.empty else _snapshot_candidates.index.max()

_current = _snapshot_candidates.loc[CURRENT_ASOF_DATE]
CURRENT_PRICE_TVL = float(_current["price"])
CURRENT_MC_TVL = float(_current["mc"])
CURRENT_TVL = float(_current["tvl"])
CURRENT_FEES_ANN = float(_current["ann_fees_30d"])
CURRENT_FEE_YIELD = float(_current["fee_yield"])
CURRENT_MULT_MAPLE = CURRENT_MC_TVL / CURRENT_FEES_ANN
CIRC_SUPPLY_TVL = CURRENT_MC_TVL / CURRENT_PRICE_TVL

print("=== Maple current snapshot (local data) ===")
print(f"Raw max date          : {pd.Timestamp(RAW_MAX_DATE).date()}")
print(f"As-of date used       : {pd.Timestamp(CURRENT_ASOF_DATE).date()}")
print(f"Price                 : ${CURRENT_PRICE_TVL:.4f}")
print(f"Market cap            : ${CURRENT_MC_TVL/1e6:.1f}M")
print(f"TVL                   : ${CURRENT_TVL/1e6:.1f}M")
print(f"Annualized fees (30d) : ${CURRENT_FEES_ANN/1e6:.1f}M")

# Weekly series for break detection
weekly_break = (
    maple_daily_tvl[["ann_fees_30d", "tvl", "fee_yield"]]
    .resample("W-WED")
    .last()
    .dropna()
)
weekly_break = weekly_break[(weekly_break["ann_fees_30d"] > 0) & (weekly_break["tvl"] > 0) & (weekly_break["fee_yield"] > 0)]
weekly_break["log_ann_fees_30d"] = np.log(weekly_break["ann_fees_30d"])
weekly_break["log_tvl"] = np.log(weekly_break["tvl"])


def detect_one_break_mean(series: pd.Series, min_side: int = 8):
    s = series.dropna().copy()
    n = len(s)
    if n < (2 * min_side + 1):
        return pd.NaT, np.nan, np.nan, np.nan

    best = None
    i_start, i_end = int(n * 0.20), int(n * 0.80)
    for i in range(max(min_side, i_start), min(n - min_side, i_end)):
        pre = s.iloc[:i]
        post = s.iloc[i:]
        pre_mean, post_mean = pre.mean(), post.mean()
        ssr = ((pre - pre_mean) ** 2).sum() + ((post - post_mean) ** 2).sum()
        pooled_var = ((pre.var(ddof=1) * (len(pre) - 1)) + (post.var(ddof=1) * (len(post) - 1))) / (len(pre) + len(post) - 2)
        if pooled_var <= 0 or not np.isfinite(pooled_var):
            strength = np.nan
        else:
            strength = abs(post_mean - pre_mean) / np.sqrt(pooled_var * (1 / len(pre) + 1 / len(post)))
        cand = (ssr, s.index[i], pre_mean, post_mean, strength)
        if (best is None) or (cand[0] < best[0]):
            best = cand
    if best is None:
        return pd.NaT, np.nan, np.nan, np.nan
    _, bdate, pre_m, post_m, strength = best
    return bdate, pre_m, post_m, strength

break_specs = [
    ("log_ann_fees_30d", "Weekly log annualized fees"),
    ("log_tvl", "Weekly log TVL"),
    ("fee_yield", "Weekly fee yield on TVL"),
]

break_rows = []
for col, lbl in break_specs:
    bdate, pre_m, post_m, strength = detect_one_break_mean(weekly_break[col])
    if pd.notna(pre_m) and pre_m != 0 and pd.notna(post_m):
        if col in {"log_ann_fees_30d", "log_tvl"}:
            pct_change = (np.exp(post_m - pre_m) - 1.0) * 100.0
        else:
            pct_change = (post_m / pre_m - 1.0) * 100.0
    else:
        pct_change = np.nan
    break_rows.append({
        "variable": lbl,
        "detected_break_date": pd.Timestamp(bdate).date() if pd.notna(bdate) else None,
        "pre_break_mean": pre_m,
        "post_break_mean": post_m,
        "pct_change_pre_to_post": pct_change,
        "break_strength_stat": strength,
        "break_date_raw": bdate,
        "series_col": col,
    })

break_summary_df = pd.DataFrame(break_rows)
print("\n=== 11B. Structural Break Detection Summary ===")
print(
    break_summary_df[[
        "variable",
        "detected_break_date",
        "pre_break_mean",
        "post_break_mean",
        "pct_change_pre_to_post",
        "break_strength_stat",
    ]].to_string(index=False, float_format="{:.4f}".format)
)

# Charts 1-3 with break lines
chart_map = {
    "log_ann_fees_30d": "maple_break_log_annualized_fees.png",
    "log_tvl": "maple_break_log_tvl.png",
    "fee_yield": "maple_break_fee_yield.png",
}
for _, r in break_summary_df.iterrows():
    col = r["series_col"]
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(weekly_break.index, weekly_break[col], linewidth=2, color="tab:blue")
    if pd.notna(r["break_date_raw"]):
        ax.axvline(r["break_date_raw"], color="tab:red", linestyle="--", linewidth=1.5)
    ax.set_title(f"{r['variable']} with detected structural break")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    out = FIG_DIR / chart_map[col]
    fig.savefig(out, dpi=170)
    plt.close(fig)

print("\nSaved charts:")
print(f"  - {FIG_DIR / 'maple_break_log_annualized_fees.png'}")
print(f"  - {FIG_DIR / 'maple_break_log_tvl.png'}")
print(f"  - {FIG_DIR / 'maple_break_fee_yield.png'}")


In [ ]:
# 11C. Post-break regime estimation
stable_period = maple_daily_tvl[
    (maple_daily_tvl.index >= pd.Timestamp("2025-10-01"))
    & maple_daily_tvl[["fee_yield", "ann_fees_30d", "tvl"]].notna().all(axis=1)
].copy()

FEE_YIELD_MEAN = float(stable_period["fee_yield"].mean())
FEE_YIELD_MEDIAN = float(stable_period["fee_yield"].median())
FEE_YIELD_STD = float(stable_period["fee_yield"].std(ddof=1))

current_fee_yield = CURRENT_FEE_YIELD
fee_yield_diff = current_fee_yield - FEE_YIELD_MEAN
fee_yield_z = fee_yield_diff / FEE_YIELD_STD if FEE_YIELD_STD > 0 else np.nan

regime_table = pd.DataFrame([
    {"metric": "Current TVL ($B)", "value": CURRENT_TVL / 1e9},
    {"metric": "Current annualized 30d fees ($M)", "value": CURRENT_FEES_ANN / 1e6},
    {"metric": "Current fee yield on TVL (%)", "value": current_fee_yield * 100},
    {"metric": "Post-break mean fee yield (%)", "value": FEE_YIELD_MEAN * 100},
    {"metric": "Post-break median fee yield (%)", "value": FEE_YIELD_MEDIAN * 100},
    {"metric": "Post-break fee-yield std (%)", "value": FEE_YIELD_STD * 100},
    {"metric": "Post-break observations (daily)", "value": float(len(stable_period))},
    {"metric": "Current vs mean fee-yield z-score", "value": fee_yield_z},
])

print("=== 11C. Post-break regime estimation ===")
print(regime_table.to_string(index=False, float_format="{:.4f}".format))

# 11D. Implied sustainable fees at current TVL
fy_low = max(0.0025, FEE_YIELD_MEAN - FEE_YIELD_STD)
fy_high = max(fy_low, FEE_YIELD_MEAN + FEE_YIELD_STD)

sustainable_fees_mean = CURRENT_TVL * FEE_YIELD_MEAN
sustainable_fees_median = CURRENT_TVL * FEE_YIELD_MEDIAN
sustainable_fees_low = CURRENT_TVL * fy_low
sustainable_fees_high = CURRENT_TVL * fy_high

fees_gap_mean_pct = (sustainable_fees_mean / CURRENT_FEES_ANN - 1.0) * 100
fees_gap_median_pct = (sustainable_fees_median / CURRENT_FEES_ANN - 1.0) * 100

print("\n=== 11D. Sustainable fee inference at current TVL ===")
print(f"Current annualized 30d fees    : ${CURRENT_FEES_ANN/1e6:.2f}M")
print(f"Sustainable fees (mean yield)  : ${sustainable_fees_mean/1e6:.2f}M ({fees_gap_mean_pct:+.1f}% vs current)")
print(f"Sustainable fees (median)      : ${sustainable_fees_median/1e6:.2f}M ({fees_gap_median_pct:+.1f}% vs current)")
print(f"Sustainable fee range          : ${sustainable_fees_low/1e6:.2f}M to ${sustainable_fees_high/1e6:.2f}M")

# Chart 4: current vs sustainable fees
fig, ax = plt.subplots(figsize=(8.5, 4.5))
labels = ["Current trailing\nannualized fees", "Sustainable\n(mean)", "Sustainable\n(median)"]
vals = [CURRENT_FEES_ANN / 1e6, sustainable_fees_mean / 1e6, sustainable_fees_median / 1e6]
ax.bar(labels, vals, color=["#7f8c8d", "#2ecc71", "#3498db"], edgecolor="white", alpha=0.9)
ax.set_ylabel("$ millions")
ax.set_title("Current vs sustainable annualized fees")
for i, v in enumerate(vals):
    ax.text(i, v + max(vals)*0.02, f"${v:.1f}M", ha="center", va="bottom", fontsize=10)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
fees_chart_path = FIG_DIR / "maple_current_vs_sustainable_fees.png"
fig.savefig(fees_chart_path, dpi=170)
plt.close(fig)

# 11E. Fundamental valuation at Maple's own current multiple (no peer multiple input)
current_multiple = CURRENT_MULT_MAPLE
implied_market_cap_mean = current_multiple * sustainable_fees_mean
implied_market_cap_median = current_multiple * sustainable_fees_median
implied_market_cap_low = current_multiple * sustainable_fees_low
implied_market_cap_high = current_multiple * sustainable_fees_high

implied_price_mean = implied_market_cap_mean / CIRC_SUPPLY_TVL
implied_price_median = implied_market_cap_median / CIRC_SUPPLY_TVL
implied_price_low = implied_market_cap_low / CIRC_SUPPLY_TVL
implied_price_high = implied_market_cap_high / CIRC_SUPPLY_TVL

price_upside_mean = (implied_price_mean / CURRENT_PRICE_TVL - 1.0) * 100
price_upside_median = (implied_price_median / CURRENT_PRICE_TVL - 1.0) * 100
price_upside_low = (implied_price_low / CURRENT_PRICE_TVL - 1.0) * 100
price_upside_high = (implied_price_high / CURRENT_PRICE_TVL - 1.0) * 100

print("\n=== 11E. Valuation at Maple's own current multiple ===")
print(f"Current multiple (MC / annualized fees): {current_multiple:.2f}x")
print(f"Implied price (mean sustainable fees)  : ${implied_price_mean:.3f} ({price_upside_mean:+.1f}%)")
print(f"Implied price (median sustainable fees): ${implied_price_median:.3f} ({price_upside_median:+.1f}%)")
print(f"Implied price range (+/-1σ fee-yield)  : ${implied_price_low:.3f} to ${implied_price_high:.3f} ({price_upside_low:+.1f}% to {price_upside_high:+.1f}%)")

# Chart 5: current price vs implied range
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.bar(["Current price", "Implied (mean)", "Implied (median)"],
       [CURRENT_PRICE_TVL, implied_price_mean, implied_price_median],
       color=["#7f8c8d", "#2ecc71", "#3498db"], edgecolor="white", alpha=0.9)
ax.errorbar([1], [implied_price_mean],
            yerr=[[implied_price_mean - implied_price_low], [implied_price_high - implied_price_mean]],
            fmt='none', ecolor='black', elinewidth=2, capsize=5)
ax.set_ylabel("Price (USD)")
ax.set_title("Current price vs implied price range at current Maple multiple")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
price_chart_path = FIG_DIR / "maple_current_vs_implied_price_range.png"
fig.savefig(price_chart_path, dpi=170)
plt.close(fig)

# Keep peer variables for later peer-validation/sanity-check sections only.
def _peer_mult_local(path):
    d = pd.read_csv(path, parse_dates=["DateTime"])
    d.columns = ["date", "price", "mc", "fdv", "fees", "circ"]
    d = d.sort_values("date").set_index("date")
    d["fees"] = d["fees"].fillna(0).clip(lower=0)
    d["ann_fees_30d"] = d["fees"].rolling(30, min_periods=10).sum() / 30 * 365
    return d[["mc", "ann_fees_30d"]].dropna()

aave_series = _peer_mult_local(DATA_DIR / "AAVE.csv")
morpho_series = _peer_mult_local(DATA_DIR / "Metric Comparison - Morpho.csv")
maple_series = maple_daily_tvl[["mc", "ann_fees_30d"]].dropna().copy()

common_peer_dates = sorted(set(maple_series.index) & set(aave_series.index) & set(morpho_series.index))
eligible_peer_dates = [d for d in common_peer_dates if d <= CURRENT_ASOF_DATE]
PEER_ASOF_DATE = eligible_peer_dates[-1] if eligible_peer_dates else CURRENT_ASOF_DATE

mult_maple_peer = float(maple_series.loc[PEER_ASOF_DATE, "mc"] / maple_series.loc[PEER_ASOF_DATE, "ann_fees_30d"])
mult_aave = float(aave_series.loc[PEER_ASOF_DATE, "mc"] / aave_series.loc[PEER_ASOF_DATE, "ann_fees_30d"])
mult_morpho = float(morpho_series.loc[PEER_ASOF_DATE, "mc"] / morpho_series.loc[PEER_ASOF_DATE, "ann_fees_30d"])
PEER_MEDIAN_MULT = float(np.median([mult_aave, mult_morpho]))
peer_median_mult = PEER_MEDIAN_MULT
discount_to_median = (mult_maple_peer / PEER_MEDIAN_MULT - 1) * 100
maple_vs_peer_median_pct = discount_to_median

peer_valuation_df = pd.DataFrame([
    ("AAVE", float(aave_series.loc[PEER_ASOF_DATE, "mc"]), float(aave_series.loc[PEER_ASOF_DATE, "ann_fees_30d"]), mult_aave, "core business-model comp"),
    ("Morpho", float(morpho_series.loc[PEER_ASOF_DATE, "mc"]), float(morpho_series.loc[PEER_ASOF_DATE, "ann_fees_30d"]), mult_morpho, "core business-model comp"),
    ("MAPLE / SYRUP", float(maple_series.loc[PEER_ASOF_DATE, "mc"]), float(maple_series.loc[PEER_ASOF_DATE, "ann_fees_30d"]), mult_maple_peer, "thesis token"),
], columns=["token", "mc", "ann_fees", "multiple", "role"])

# Keep scenario placeholders for downstream notebook cells.
bear_r = {"implied_price": implied_price_low, "return_pct": price_upside_low}
base_r = {"implied_price": implied_price_mean, "return_pct": price_upside_mean}
bull_r = {"implied_price": implied_price_high, "return_pct": price_upside_high}

# Compact section summary block
break_dates_compact = {
    row["variable"]: row["detected_break_date"] for _, row in break_summary_df.iterrows()
}
print("\n=== Compact Structural-Break Summary ===")
print(f"Current as-of date                : {pd.Timestamp(CURRENT_ASOF_DATE).date()}")
print(f"Detected break dates              : {break_dates_compact}")
print(f"Post-break mean fee yield         : {FEE_YIELD_MEAN*100:.2f}%")
print(f"Current fee yield                 : {CURRENT_FEE_YIELD*100:.2f}%")
print(f"Current annualized fees (30d)     : ${CURRENT_FEES_ANN/1e6:.2f}M")
print(f"Sustainable annualized fees (mean): ${sustainable_fees_mean/1e6:.2f}M")
print(f"Maple own current multiple        : {current_multiple:.2f}x")
print(f"Implied price mean                : ${implied_price_mean:.3f}")
print(f"Implied price range               : ${implied_price_low:.3f} to ${implied_price_high:.3f}")
print("Saved charts:")
print(f"  - {fees_chart_path}")
print(f"  - {price_chart_path}")

In [ ]:
# 11F. Interpretation / investment implication
print("=== 11F. Interpretation / Investment Implication ===")
print(
    "The break tests suggest Maple's operating profile transitioned across late 2024 into 2025, "
    "with scale and monetization variables shifting at different times."
)
print(
    "The relevant valuation anchor is the post-break business rather than the full low-scale history; "
    "current trailing fees appear below what the post-break fee-yield regime would imply at current TVL."
)
print(
    "Even at Maple's own current multiple, the resulting implied price range supports moderate upside versus spot "
    "and is consistent with an independent fundamental support layer before peer validation."
)

# Optional exploratory appendix: Monte Carlo around Maple-only post-break parameters.
N_SIM = 10000
rng = np.random.default_rng(42)

tvl_log_sigma = float(np.log(maple_daily_tvl["tvl"].dropna()).diff().dropna().std())
tvl_sigma_6m = tvl_log_sigma * np.sqrt(180)
sim_tvl = CURRENT_TVL * np.exp(rng.normal(-0.5 * tvl_sigma_6m**2, tvl_sigma_6m, N_SIM))
sim_yield = rng.normal(FEE_YIELD_MEAN, FEE_YIELD_STD if FEE_YIELD_STD > 0 else 1e-6, N_SIM).clip(0.0025, 0.10)
sim_fees = sim_tvl * sim_yield
sim_mc = CURRENT_MULT_MAPLE * sim_fees
sim_price = sim_mc / CIRC_SUPPLY_TVL

print("\n=== Appendix (Exploratory): Maple-only Monte Carlo ===")
print(f"Median implied price: ${np.median(sim_price):.4f}")
print(f"10th-90th pct range : ${np.percentile(sim_price,10):.4f} to ${np.percentile(sim_price,90):.4f}")
print("(Exploratory only; main thesis uses deterministic structural-break outputs.)")


In [ ]:
# 11 appendix (exploratory only): Maple-only Monte Carlo around post-break regime
N_SIM = 10_000
RNG = np.random.default_rng(42)

tvl_series = maple_daily_tvl["tvl"].dropna()
tvl_daily_sigma = float(np.log(tvl_series).diff().dropna().std()) if len(tvl_series) > 3 else 0.0
tvl_6m_sigma = tvl_daily_sigma * np.sqrt(180)

sim_tvl = CURRENT_TVL * np.exp(RNG.normal(-0.5 * tvl_6m_sigma**2, tvl_6m_sigma, N_SIM))
sim_fee_yield = RNG.normal(FEE_YIELD_MEAN, FEE_YIELD_STD if FEE_YIELD_STD > 0 else 1e-6, N_SIM).clip(0.0025, 0.10)
sim_fees = sim_tvl * sim_fee_yield
sim_mc = CURRENT_MULT_MAPLE * sim_fees
sim_price = sim_mc / CIRC_SUPPLY_TVL

print("=== Appendix (Exploratory): Maple-only Monte Carlo ===")
print(f"Median implied price : ${np.median(sim_price):.4f}")
print(f"10th pct implied     : ${np.percentile(sim_price,10):.4f}")
print(f"90th pct implied     : ${np.percentile(sim_price,90):.4f}")
print("Use as exploratory context only; deterministic structural-break outputs remain primary.")

### 11F. Interpretation / Investment Implication

The break tests suggest Maple's operating profile transitioned across late 2024 into 2025, with scale and monetization variables shifting at different times. The relevant valuation anchor is therefore the post-break business, not full-history averages dominated by earlier low-scale periods.

Within the post-break calibration regime, current TVL supports sustainable annualized fees above the current trailing 30d run-rate in mean and median cases. This is consistent with the view that current trailing fees may understate Maple's sustainable earning capacity at current scale.

Even if Maple is valued only at its own current MC / annualized-fees multiple, the post-break sustainable-fee estimates support moderate upside versus spot in central cases. This supports an independent fundamental layer before peer-validation overlays are introduced.

## 12. Peer Relative Valuation (Aave & Morpho)

Aave and Morpho are the only core business-model comparables used in this notebook. This section keeps the peer set narrow and interpretable, then maps Maple's factor-loading and multiple context relative to those two lending peers only.

| Token | Role | Rationale |
|-------|------|-----------|
| **AAVE** | Core business-model comp | Large incumbent DeFi lender |
| **MORPHO** | Core business-model comp | Fast-growing lending protocol |
| **MAPLE / SYRUP** | Thesis token | Target asset |

Use this section to benchmark Maple's positioning versus the core peer set, not to claim perfect comparability across all protocol designs.

In [ ]:
def analyze_token(token_symbol, label=None):
    """Run 4-factor regression for any token and return a summary dict."""
    label = label or token_symbol.upper()
    df = weekly_df[weekly_df["asset"] == token_symbol].copy().sort_values("date")
    if len(df) < 20:
        print(f"⚠  {label}: only {len(df)} rows – skipping")
        return None

    df["raw_ret"] = df["price"].pct_change()
    df["log_ret"] = np.log(df["price"]).diff()

    merged = (
        df.set_index("date")[["raw_ret", "log_ret"]]
        .join(factor_returns_df.set_index("date")[["market", "smb", "mom", "value"]], how="inner")
        .dropna()
    )
    if len(merged) < 20:
        print(f"⚠  {label}: only {len(merged)} merged rows after dropna – skipping")
        return None

    X = sm.add_constant(merged[["market", "smb", "mom", "value"]])
    model_log = sm.OLS(merged["log_ret"], X).fit()

    latest_mc = df["mc"].dropna().iloc[-1] if df["mc"].notna().any() else np.nan

    result = {
        "token": label,
        "obs": len(merged),
        "beta_mkt": model_log.params["market"],
        "beta_smb": model_log.params["smb"],
        "beta_mom": model_log.params["mom"],
        "beta_value": model_log.params["value"],
        "alpha_ann": annualize_from_weekly_log(model_log.params["const"], model_log.bse["const"])[1],
        "r2": model_log.rsquared,
        "p_mkt": model_log.pvalues["market"],
        "p_smb": model_log.pvalues["smb"],
        "p_value": model_log.pvalues["value"],
        "mc_M": latest_mc / 1e6 if pd.notna(latest_mc) else np.nan,
    }
    return result

# ── Run for each comparable ──────────────────────────────────
COMPARABLES = {
    MAPLE_SYMBOL: "MAPLE / SYRUP",
    "aave": "AAVE",
    "morpho": "MORPHO",
}

comp_results = {}
for sym, lbl in COMPARABLES.items():
    print(f"Analysing {lbl} ({sym})...")
    r = analyze_token(sym, lbl)
    if r:
        comp_results[sym] = r

print("\nDone.")


In [ ]:
# ── Comparison table ─────────────────────────────────────────
if comp_results:
    comp_df = pd.DataFrame(comp_results).T[[
        "token", "obs", "beta_mkt", "beta_smb", "beta_value", "alpha_ann", "r2", "mc_M"
    ]].rename(columns={
        "beta_mkt": "β_MKT",
        "beta_smb": "β_SMB",
        "beta_value": "β_VALUE",
        "alpha_ann": "α_ann (simple)",
        "r2": "R²",
        "mc_M": "MC ($M)",
    })

    # Pull valuation columns from Section 11 source-of-truth table (MC / annualized 30d fees).
    valuation_map = {}
    if 'peer_valuation_df' in globals() and not peer_valuation_df.empty:
        for _, row in peer_valuation_df.iterrows():
            token = row["token"]
            valuation_map[token] = {
                "MC/AnnFees (30d)": row["multiple"],
                "Ann. Fees ($M)": row["ann_fees"] / 1e6 if pd.notna(row["ann_fees"]) else np.nan,
            }

    comp_df["MC/AnnFees (30d)"] = comp_df["token"].map(lambda t: valuation_map.get(t, {}).get("MC/AnnFees (30d)", np.nan))
    comp_df["Ann. Fees ($M)"] = comp_df["token"].map(lambda t: valuation_map.get(t, {}).get("Ann. Fees ($M)", np.nan))

    numeric_cols = ["β_MKT", "β_SMB", "β_VALUE", "α_ann (simple)", "R²", "MC ($M)", "MC/AnnFees (30d)", "Ann. Fees ($M)"]
    comp_df[numeric_cols] = comp_df[numeric_cols].apply(pd.to_numeric, errors="coerce")

    print("=== Comparable Projects — Factor Loading Comparison (Section 11 valuation basis) ===")
    print(comp_df.to_string(float_format="{:.3f}".format))


In [ ]:
# ── Visualise factor loadings ────────────────────────────────
if comp_results:
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    factors_plot = ["β_MKT", "β_SMB", "β_VALUE"]
    colors = ["steelblue","darkorange","forestgreen","crimson"]

    for ax, fac in zip(axes, factors_plot):
        values = comp_df[fac].values
        tokens = comp_df["token"].values
        bars   = ax.bar(tokens, values, color=colors[:len(tokens)], alpha=0.8, edgecolor="white")
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_title(f"{fac} Loading")
        ax.set_ylabel("Factor Beta")
        for bar, v in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.02*np.sign(v),
                    f"{v:.2f}", ha="center", va="bottom" if v >= 0 else "top", fontsize=9)

    plt.suptitle("Factor Beta Comparison: MAPLE vs DeFi Lending Peers", fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()


## 13. Why Now — Scenario Valuation Framing

The near-term setup is framed through deterministic bear/base/bull scenario ranges. The core idea is straightforward: if Maple sustains a higher fee regime tied to a large TVL base, then valuation upside can come from partial rerating toward core peer multiples.

This section uses only the Section 11 valuation basis: `MC / annualized 30d fees` with daily current snapshot inputs and explicit scenario assumptions.

In [ ]:
# ── Section 13 source-of-truth summary (from Section 11 framework) ─────────
#
# This cell intentionally avoids re-computing valuation via mc_fees_ratio.
# All valuation outputs are inherited from Section 11 daily snapshot +
# MC/annualized-30d-fees framework.

peers = peer_valuation_df[peer_valuation_df["token"] != "MAPLE / SYRUP"].copy()
val_df = peer_valuation_df.copy()
val_df["Current multiple"] = val_df["multiple"]
val_df["Peer median multiple"] = PEER_MEDIAN_MULT
val_df["Implied upside/downside %"] = (val_df["Peer median multiple"] / val_df["Current multiple"] - 1) * 100
val_df["Interpretation label"] = val_df["Implied upside/downside %"].apply(
    lambda x: "modest" if x > 10 else ("neutral" if x >= -10 else "stretched")
)

pt_median = base_r["implied_price"]
maple_mult = CURRENT_MULT_MAPLE
maple_fees = CURRENT_FEES_ANN / 1e6
latest_maple_price = CURRENT_PRICE_TVL
latest_maple_mc = CURRENT_MC_TVL / 1e6
implied_upside_pct = (pt_median / latest_maple_price - 1) * 100

print("=== MAPLE / SYRUP Valuation Summary (Section 11 aligned) ===")
print(f"Current snapshot as-of    : {pd.Timestamp(CURRENT_ASOF_DATE).date()}")
print(f"Peer comp as-of           : {pd.Timestamp(PEER_ASOF_DATE).date()}")
print(f"Current price             : ${latest_maple_price:.4f}")
print(f"Current MC                : ${latest_maple_mc:.1f}M")
print(f"Current multiple          : {maple_mult:.2f}x (MC / annualized 30d fees)")
print(f"Annualized fees (30d)     : ${maple_fees:.1f}M")
print(f"Aave/Morpho median mult   : {PEER_MEDIAN_MULT:.2f}x")
print(f"Maple vs peer median      : {discount_to_median:+.1f}%")
print(f"Base-case implied price   : ${pt_median:.4f} ({implied_upside_pct:+.1f}%)")

print("\n=== Multiple Comparison Table ===")
print(
    val_df[[
        "token", "Current multiple", "Peer median multiple", "Implied upside/downside %", "Interpretation label"
    ]].to_string(float_format="{:.2f}".format, index=False)
)


In [ ]:
# ── Scenario valuation chart (Section 11 aligned labels) ──────────────────
cases = {
    "Bear\n(lower TVL, current yield,\nno re-rating)": bear_r["implied_price"],
    "Base\n(current TVL, current yield,\nAave/Morpho median)": base_r["implied_price"],
    "Bull\n(higher TVL, stable yield,\nMorpho-like multiple)": bull_r["implied_price"],
}

fig, ax = plt.subplots(figsize=(10, 4.5))
colors_cases = ["#e74c3c", "#2ecc71", "#3498db"]
bars = ax.bar(cases.keys(), cases.values(), color=colors_cases, alpha=0.85, edgecolor="white", width=0.58)
ax.axhline(latest_maple_price, color="black", linewidth=1.5, linestyle="--", label=f"Current price ${latest_maple_price:.3f}")
for bar, v in zip(bars, cases.values()):
    pct = (v / latest_maple_price - 1) * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        v + 0.005,
        f"${v:.3f}\n({pct:+.0f}%)",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )
ax.set_ylabel("Price (USD)")
ax.set_title("MAPLE / SYRUP TVL-Anchored Scenario Valuation")
ax.legend()
plt.tight_layout()
plt.show()

print("\n=== Scenario Range Summary (source-of-truth from Section 11) ===")
for label, price in cases.items():
    clean_label = label.replace("\n", " ")
    print(f"  {clean_label:55s}: ${price:.4f}  ({(price/latest_maple_price-1)*100:+.1f}%)")


## 14. Conclusion for March 18, 2026 Thesis Use

MAPLE / SYRUP supports a credible **long small-cap thesis** when framed through fundamentals, peer-relative valuation, and TVL-anchored fee-regime stability.

- **Main valuation takeaway 1:** Maple's fee generation appears supported by a large asset base and a relatively stable fee yield.
- **Main valuation takeaway 2:** Maple still trades below the Aave/Morpho peer multiple range on current MC / annualized 30d fees.
- **Main valuation takeaway 3:** The upside case comes from sustained fee generation plus partial or full peer rerating.
- **Kalman note (appendix):** state-space outputs are supporting context (neutral vs own regime), not the primary upside engine.
- **Decision framing:** use the deterministic Section 11 scenario table as the main-body valuation anchor.

In [ ]:
# ── Final summary stats ──────────────────────────────────────
print("=== MAPLE / SYRUP Factor Model Summary ===")
compact_summary(model_log, "4-Factor Log Returns")

latest_maple_price = CURRENT_PRICE_TVL

print("\n=== Key Metrics (TVL-Anchored Framework) ===")
print(f"Current snapshot as-of       : {pd.Timestamp(CURRENT_ASOF_DATE).date()}")
print(f"Peer comparison as-of        : {pd.Timestamp(PEER_ASOF_DATE).date()}")
print(f"Current Price                : ${latest_maple_price:.4f}")
if 'CURRENT_TVL' in globals() and pd.notna(CURRENT_TVL):
    print(f"Current TVL                   : ${CURRENT_TVL/1e6:.0f}M")
print(f"Current annualized 30d fees   : ${CURRENT_FEES_ANN/1e6:.1f}M")
print(f"Maple MC/fees multiple        : {CURRENT_MULT_MAPLE:.2f}x")
print(f"Aave MC/fees multiple         : {mult_aave:.2f}x")
print(f"Morpho MC/fees multiple       : {mult_morpho:.2f}x")
print(f"Aave/Morpho median multiple   : {peer_median_mult:.2f}x")
print(f"Maple discount vs peer median : {maple_vs_peer_median_pct:+.1f}%")
print("Kalman status (appendix note) : neutral vs own regime (supporting context)")
if 'base_r' in globals() and latest_maple_price > 0:
    print(
        f"Base-case scenario (main body): ${base_r['implied_price']:.4f} "
        f"({(base_r['implied_price']/latest_maple_price-1)*100:+.1f}%)"
    )

## 15. Automated Verification Checks (Traffic-Light Output)

This block reports four buckets separately:
1. Data integrity checks
2. Factor-model checks
3. TVL-anchored fee-yield checks
4. Valuation-comparable checks

Traffic-light convention:
- **Green** = safe to use
- **Yellow** = usable with caveats
- **Red** = do not rely on in final memo

In [ ]:
def bucket_status(all_pass, any_fail):
    if all_pass:
        return "Green"
    if any_fail:
        return "Red"
    return "Yellow"

# 1) Data integrity
data_checks = [
    ("Data pull non-empty", len(pivoted_df) > 0, f"rows={len(pivoted_df)}"),
    ("Weekly sample non-empty", len(weekly_df) > 0, f"rows={len(weekly_df)}"),
    ("MAPLE/SYRUP symbol matched", MAPLE_SYMBOL in set(weekly_df['asset'].dropna().unique()), MAPLE_SYMBOL),
]

data_df = pd.DataFrame(data_checks, columns=["check", "passed", "detail"])

# 2) Factor-model checks
factor_checks = [
    ("Factor observations >= 52", len(factor_returns_df) >= 52, f"n={len(factor_returns_df)}"),
    ("Regression observations >= 52", len(reg_df) >= 52, f"n={len(reg_df)}"),
    ("R^2 below overfit concern threshold", float(r2_log) < 0.60, f"R^2={r2_log:.3f}"),
]
factor_df = pd.DataFrame(factor_checks, columns=["check", "passed", "detail"])

# 3) TVL-anchored fee-yield checks
fee_yield_checks = [
    ("TVL metric available", bool(TVL_AVAILABLE), "available" if TVL_AVAILABLE else "missing"),
    (
        "Stable fee-yield sample size >= 60 daily obs",
        int(len(stable_period)) >= 60 if 'stable_period' in globals() else False,
        f"n={len(stable_period) if 'stable_period' in globals() else 0}",
    ),
    (
        "Fee-yield mean in plausible range (1% to 5%)",
        (0.01 <= float(FEE_YIELD_MEAN) <= 0.05) if 'FEE_YIELD_MEAN' in globals() and pd.notna(FEE_YIELD_MEAN) else False,
        f"mean={float(FEE_YIELD_MEAN)*100:.2f}%" if 'FEE_YIELD_MEAN' in globals() and pd.notna(FEE_YIELD_MEAN) else "nan",
    ),
]
fee_yield_df = pd.DataFrame(fee_yield_checks, columns=["check", "passed", "detail"])

# 4) Valuation-comp checks
peer_n = len(peers) if 'peers' in globals() else 0
maple_row_exists = bool(
    'val_df' in globals()
    and isinstance(val_df, pd.DataFrame)
    and not val_df[val_df['token'] == 'MAPLE / SYRUP'].empty
)
valuation_checks = [
    ("Peer set size >= 2", peer_n >= 2, f"peer_n={peer_n}"),
    ("MAPLE / SYRUP valuation row exists", maple_row_exists, "present" if maple_row_exists else "missing"),
    (
        "Maple discount vs peer median computed",
        'maple_vs_peer_median_pct' in globals() and pd.notna(maple_vs_peer_median_pct),
        f"{maple_vs_peer_median_pct:+.1f}%" if 'maple_vs_peer_median_pct' in globals() and pd.notna(maple_vs_peer_median_pct) else "nan",
    ),
]
valuation_df = pd.DataFrame(valuation_checks, columns=["check", "passed", "detail"])

print("=== Verification: Data Integrity ===")
print(data_df.to_string(index=False))
print("\n=== Verification: Factor Model ===")
print(factor_df.to_string(index=False))
print("\n=== Verification: TVL-Anchored Fee Yield ===")
print(fee_yield_df.to_string(index=False))
print("\n=== Verification: Valuation Comps ===")
print(valuation_df.to_string(index=False))

data_light = bucket_status(bool(data_df['passed'].all()), bool((~data_df['passed']).any()))
factor_light = bucket_status(bool(factor_df['passed'].all()), bool((~factor_df['passed']).any()))
fee_yield_light = bucket_status(bool(fee_yield_df['passed'].all()), bool((~fee_yield_df['passed']).any()))
valuation_light = bucket_status(bool(valuation_df['passed'].all()), bool((~valuation_df['passed']).any()))

summary_df = pd.DataFrame([
    {"bucket": "Data integrity", "traffic_light": data_light},
    {"bucket": "Factor-model checks", "traffic_light": factor_light},
    {"bucket": "TVL-anchored fee-yield checks", "traffic_light": fee_yield_light},
    {"bucket": "Valuation-comp checks", "traffic_light": valuation_light},
])

print("\n=== Final Traffic-Light Summary ===")
print(summary_df.to_string(index=False))